In [1]:

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Audio
import lightning as L
import sys
from lightning.pytorch.callbacks import LearningRateMonitor

sys.path.append('../')
sys.path.append('./')
import importlib
import yaml
import torch
from tqdm.auto import tqdm

## Init simple dataset for audioset task

In [2]:

import h5py
import torch
import glob
import pickle
import numpy as np

import pandas as pd

class jsinv3_audioset_SSL(torch.utils.data.ConcatDataset):
    # Makes a dataset using pre-paired speech and audioset background sounds
    # Works with hdf5 files for the jsinv3 dataset.
    hdf5_glob = "JSIN_all__run_*.h5"
    target_keys = ["noise/labels_binary_via_int"]

    def __init__(
        self, root, train=True, download=False, transform=None, batch_size=1, eval_max=3
    ):
        """
        Builds the pytorch hdf5 combined dataset from the files found in the
        specified root directory.
        """
        del download

        if train:
            self.all_hdf5_files = glob.glob(root + "/train_*/" + self.hdf5_glob)
        else:
            if eval_max == -1:
                self.all_hdf5_files = glob.glob(root + "/valid_*/" + self.hdf5_glob)
            else:
                self.all_hdf5_files = glob.glob(root + "/valid_*/" + self.hdf5_glob)[
                    0:eval_max
                ]

        self.datasets = [
            H5DatasetAudiosetSSL(h5_file, transform, self.target_keys, batch_size, train=train)
            for h5_file in self.all_hdf5_files
        ]

        super().__init__(self.datasets)
        self.rotate_index = 0


    def class_map(self):
        """
        Loads the mapping between the word IDX and human readable word map.
        """
        word_and_speaker_encodings = pickle.load(
            open("word_and_speaker_encodings_jsinv3.pckl", "rb")
        )
        class_map = word_and_speaker_encodings["word_idx_to_word"]
        return class_map



class H5DatasetAudiosetSSL(torch.utils.data.Dataset):
    def __init__(self, path, transform, target_keys, batch_size, train=True):
        """
        Builds a pytorch hdf5 dataset. Returns signal1, signal2, label if train; signal1, label else
        Args:
            path (str): location of the hdf5 dataset
        """
        self.file_path = path
        self.dataset = None
        self.transform = transform
        self.target_keys = target_keys
        self.batch_size = batch_size
        self.train = train

        # These TODOs are not implemented for the release. HDF5 files are
        # already shuffled, so we can run through them directly.
        # TODO: implement chunking the hdf5 file so that we can shuffle the data
        # TODO: implement shuffling the audioset and the speech separately
        # self.chunk_size = hdf5_chunk_size
        with h5py.File(self.file_path, "r", swmr=True) as file:
            self.n_signals = len(file["sources"]["signal"]["signal"])

        # scale dataset len after setting split indices
        self.dataset_len =  self.n_signals // self.batch_size
          # scale by batch size for dataloader (accessed in len method)

    def __getitem__(self, index):
        """
        """
        if self.dataset is None:
            self.dataset = h5py.File(
                self.file_path, "r", swmr=True
            )  # ["ndarray_data"]["signal"]
        # set up ix logic
        start = index * self.batch_size
        end = start + self.batch_size # second half will be noises used for augmentation

        batch_ixs = np.arange(start, end)
        noises = self.dataset["sources"]["noise"]["signal"][batch_ixs]

        if self.train:
            aug_ix = np.random.randint(self.n_signals - self.batch_size) # second half will be noises used for augmentation
            augments = self.dataset["sources"]["noise"]["signal"][aug_ix: aug_ix + self.batch_size ]
        ## Get labels 
        target_paths = self.target_keys[0].split("/")
        target = self.dataset["sources"][target_paths[0]][target_paths[1]][
                batch_ixs
            ]

        # get where noises are None 
        bad_ixs = np.argwhere(noises.sum(1) == 0).flatten()
        if len(bad_ixs) > 0:
            good_ixs = batch_ixs[~np.isin(batch_ixs, bad_ixs)]
            new_samples = np.random.choice(good_ixs, size=len(bad_ixs)+2)[:len(bad_ixs)] 
            samp_ixs = np.argwhere(np.isin(batch_ixs, new_samples)).flatten()
            assert len(bad_ixs) == len(samp_ixs), f"{len(bad_ixs)} bad ixs, but drew {len(samp_ixs)} ixs to resample."
            noises[bad_ixs] = noises[samp_ixs]
            target[bad_ixs] = target[samp_ixs]
        
        target = torch.from_numpy(target).float()
        
        if self.train:
            aud_1 = []
            aud_2 = []
            for noise in noises:
                aug_ixs = np.random.randint(self.batch_size, size=2)
                aug_1, aug_2 = augments[aug_ixs]
                # get view
                view1, _ = self.transform(noise, aug_1)
                view2, _ = self.transform(noise, aug_2)
                aud_1.append(view1)
                aud_2.append(view2)
            aud_1 = torch.stack(aud_1)
            aud_2 = torch.stack(aud_2)

            return aud_1, aud_2, target 
        else:
            aud_1 = torch.stack([
                self.transform(noise, None)[0] for noise in noises
            ])
            return aud_1, target 
        
    def __len__(self):
        return self.dataset_len

## Init min lightning module 

In [6]:

import torch
from torch import nn
# import torchvision
import torch.nn.functional as F
import lightning as L
import os, sys

sys.path.append(os.path.join(os.path.abspath(os.getcwd()), "lightning_scripts"))
import architectures
from torchmetrics.classification import Accuracy, BinaryPrecision

import losses as ssl_losses
# import audio_ssl.losses as ssl_losses 

from audio_ssl.misc import LARS, CosineWarmupScheduler
from typing import List, Union, Tuple
# from pprint import pprint

# from jsinV3DataLoader_precombined_batched import jsinV3_precombined_paired_batched
import robustness.audio_functions.audio_transforms as at 
from robustness.audio_functions.audio_input_representations import AUDIO_INPUT_REPRESENTATIONS

class ModelWithFrontEnd(nn.Module):
    def __init__(self,front_end, model):
        super().__init__()
        self.front_end = front_end
        self.model = model

    def forward(self, x, with_latent=False, fake_relu=False, no_relu=False):
        x, _ = self.front_end(x, None)
        if with_latent:
            return self.model.f(x,  with_latent=with_latent, fake_relu=fake_relu, no_relu=no_relu)
        else:
            feature, out, logits = self.model(x)
            return feature, out, logits    


class LitAudioSetSSL(L.LightningModule):
    def __init__(self, config):
        super().__init__()
        self.save_hyperparameters()
        self.config = config 

        # Init audio transforms 
        self.transforms = at.AudioCompose([
                at.AudioToTensor(),
                at.__dict__[self.config['audio_transforms']['crop']](**self.config['audio_transforms']['crop_kwrgs']),
                at.CombineWithRandomDBSNR(low_snr=config['audio_transforms']['low_snr'],
                                        high_snr=config['audio_transforms']['high_snr']),
                at.DBSPLNormalizeForegroundAndBackground(dbspl=config['audio_transforms']['dbspl']),
                at.UnsqueezeAudio(dim=0) # dim=0 here so batches of audio from dataloader will be (Batch, 1, Time)
            ])

        # Get audio config and init representation 
        self.audio_config = AUDIO_INPUT_REPRESENTATIONS[config['audio_rep']['name']]
        self.audio_rep = at.AudioToAudioRepresentation(**self.audio_config)

        # Get audio model from config kwargs
        self.model = architectures.__dict__[self.config['model']['arch_name']](**self.config['model']['arch_kwargs'])
        self.metamer_layers = [
            'input_after_preproc',
            'conv1',
            'bn1',
            'conv1_relu1',
            'maxpool1',
            'layer1',
            'layer2',
            'layer3',
            'layer4',
            'avgpool',
        ]


        self.model = ModelWithFrontEnd(self.audio_rep, self.model)
                # init losses 
        # if torch.distributed.is_initialized():
        self.distributed = torch.distributed.is_initialized()
        self.ssl_task = self.config['hparas']['ssl_task']
        self.ssl_loss = self.get_loss()

        # scaling factor to apply to self-supervised task loss - default is 1.
        self.lambda_ssl = self.config['hparas'].get('lambda_ssl', 1.0)
        self.opt_supervised_task = self.config['model']['arch_kwargs']['supervised']
        if self.opt_supervised_task:
            self.class_loss = nn.BCEWithLogitsLoss()
            self.calc_precision = BinaryPrecision()

        # get lower bound for MMCR task 

    def _step(self, batch, batch_idx, step_type):
        aud_1, aud_2, labels  = batch
        # pass pairs through model 
        _, out_1, logits_1 = self.model(aud_1)
        _, out_2, logits_2 = self.model(aud_2)

        loss_ssl = self.ssl_loss(out_1, out_2) 

        self.log(f"{step_type}_{self.ssl_loss_str}_loss", loss_ssl.detach(), on_step=True, on_epoch=False, prog_bar=True, sync_dist=True)

        class_loss = 0.0
        if self.opt_supervised_task:
        # get classification loss
            class_loss_1 = self.class_loss(logits_1, labels)
            class_loss_2 = self.class_loss(logits_2, labels)
            class_loss = (class_loss_1 + class_loss_2 ) / 2.0

            # calc acc 
            prec = 0 
            prec += self.calc_precision(logits_1, labels).item()
            prec += self.calc_precision(logits_2, labels).item()
            prec /= 2.0

            self.log(f"{step_type}_class_prec", prec, on_step=True, on_epoch=True, prog_bar=True, sync_dist=True)
            self.log(f"{step_type}_class_loss", class_loss.detach(), on_step=True, on_epoch=False, prog_bar=True, sync_dist=True)

        total_loss = self.lambda_ssl * loss_ssl + class_loss

        self.log(f"{step_type}_total_loss", total_loss.detach(), on_step=True, on_epoch=True, prog_bar=True, sync_dist=True)

        if 'mmcr' in self.ssl_loss_str:
            ppe = (self.mmcr_lower_bound + loss_ssl.detach()) / self.mmcr_lower_bound
            self.log(f"{step_type}_ppe", ppe, on_step=True, on_epoch=True, prog_bar=True, sync_dist=True)

        return total_loss

    def training_step(self, batch, batch_idx):
        return self._step(batch, batch_idx, "train")

    def _eval_step(self, batch, batch_idx, step_type):
        # Test step only for JSIN eval 
        aud, labels  = batch
        # pass pairs through model 
        _, out_1, logits_1 = self.model(aud)
        
        class_loss = self.class_loss(logits_1, labels)
        prec = self.calc_precision(logits_1, labels)

        self.log(f"{step_type}_class_prec", prec, on_step=True, on_epoch=True, prog_bar=True, sync_dist=True)
        self.log(f"{step_type}_class_loss", class_loss.detach(), on_step=True, on_epoch=True, prog_bar=True, sync_dist=True)
        return class_loss
    
    def validation_step(self, batch, batch_idx):
        return self._eval_step(batch, batch_idx, "val")

    def test_step(self, batch, batch_idx):
        return self._eval_step(batch, batch_idx, "test")

    def configure_optimizers(self):
        # Optimizer
        if self.config['hparas']['optimizer'] == "LARS":
            if self.config['hparas'].get("num_warmup_steps_or_ratio", False):
                # Typical learning rate scheduling is handled in CosineWarmupScheduler
                # as init_lr * batchsize / 256
                # lr given to LARS is 0
                #  CosineWarmupScheduler handles incrementing the LR
                self.optimizer = LARS(
                                self.model.parameters(),
                                lr=0,
                                weight_decay=1e-6,
                                momentum=0.9,
                                weight_decay_filter=True,
                                lars_adaptation_filter=True,
                            )
                total_training_steps = self.total_training_steps()
                num_warmup_steps = self.compute_warmup(total_training_steps, self.config['hparas']['num_warmup_steps_or_ratio'])
                lr_scheduler = CosineWarmupScheduler(
                    optimizer=self.optimizer,
                    batch_size=self.config['hparas']['global_batch_size'], # is global batch size
                    warmup_steps=num_warmup_steps,
                    max_steps=total_training_steps,
                    lr=self.config['hparas']['lr']
                )
                return [self.optimizer], [
                    {
                        'scheduler': lr_scheduler,  # The LR scheduler instance (required)
                        'interval': 'step',  # The unit of the scheduler's step size
                    }
                ]                  
            else:
                lr = self.config['hparas']['lr'] * self.config['hparas']['global_batch_size'] / 256 
                self.optimizer = LARS(
                                self.model.parameters(),
                                lr=lr,
                                weight_decay=1e-5,
                                momentum=0.9,
                                weight_decay_filter=True,
                                lars_adaptation_filter=True,
                            )                        
        else:
            lr = self.config['hparas']['lr'] * self.config['hparas']['global_batch_size'] / 256 
            opt = getattr(torch.optim, self.config['hparas']['optimizer'])
            self.optimizer = opt(self.model.parameters(), lr=lr)      
        return [self.optimizer]

    def on_before_optimizer_step(self, _):
        def _get_grad_norm(params, scale=1):
            """Compute grad norm given a gradient scale."""
            total_norm = 0.0
            for p in params:
                if p.grad is not None:
                    param_norm = (p.grad.detach().data / scale).norm(2)
                    total_norm += param_norm.item() ** 2
            total_norm = total_norm**0.5
            return total_norm
        grad_norm = _get_grad_norm(self.model.parameters())
        self.log("grad_norm", torch.tensor(grad_norm), prog_bar=True, on_step=True, on_epoch=False)

    def forward(self, x):
        """
        PL required forward wrapper. Enables calling model in two ways:
        1) standard call in .py scripts
            model = LitAudioSSL(args)
            outs = model(inputs)
        2) inside this lightning module's methods as self (eg in _step)
            outs = self(inputs) # self is self.forward, and is same as self.model.forward 
        """
        return self.model(x)
    
    def collate_fn(self, batch):
        batch = batch[0]
        return batch 


    def train_dataloader(self):
        # set train dataloader as attr so we can rotate examples every epoch 
        dataset = jsinv3_audioset_SSL(root=self.config['data']['root'],
                                            train=True,
                                            batch_size=self.config['hparas']['batch_size'],
                                            transform=self.transforms)
        train_dataloader = torch.utils.data.DataLoader(
            dataset,
            batch_size=1,
            num_workers=self.config['num_workers'], 
            pin_memory=True,
            # persistent_workers=True,
            shuffle=False,
            collate_fn=self.collate_fn
        )
        return train_dataloader
    
    def val_dataloader(self):
        dataset = jsinv3_audioset_SSL(root=self.config['data']['root'],
                                            train=False,
                                            batch_size=self.config['hparas']['batch_size'],
                                            transform=self.transforms)
        dataloader = torch.utils.data.DataLoader(
            dataset,
            batch_size=1,
            num_workers=self.config['num_workers'],
            shuffle=False,
            collate_fn=self.collate_fn
        )
        return dataloader

    # @property
    def total_training_steps(self) -> int:
        dataset_size = len(self.train_dataloader())
        num_devices = self.config['num_gpus']
        effective_batch_size = self.trainer.accumulate_grad_batches * num_devices
        max_estimated_steps = (dataset_size // effective_batch_size) * self.trainer.max_epochs

        if self.trainer.max_steps and self.trainer.max_steps < max_estimated_steps and self.trainer.max_steps != -1:
            return int(self.trainer.max_steps)
        return int(max_estimated_steps)

    def compute_warmup(self, num_training_steps: int, num_warmup_steps: Union[int, float]) -> int:
        return num_warmup_steps * num_training_steps if isinstance(num_warmup_steps, float) else num_warmup_steps
    
    @property
    def mmcr_lower_bound(self) -> int:
        # precompute mmcr lower bound as prop 3.3 from https://arxiv.org/pdf/2406.09366
        p = torch.tensor(self.config['hparas']['global_batch_size'])
        d = torch.tensor(self.config['model']['arch_kwargs']['projector_dims'][-1])
        return torch.sqrt(p * torch.min(p, d))

    def get_loss_fn(self, loss_fn_name, loss_kwargs):
        loss_fn = ssl_losses.__dict__[loss_fn_name]
        return loss_fn(**loss_kwargs, distributed=self.distributed) if loss_kwargs else loss_fn(distributed=self.distributed)

    def get_loss(self):
        self.ssl_loss_str = self.config['hparas']['ssl_loss_str'] # str for logs 
        loss_kwargs = self.config['hparas'].get('ssl_loss_kwargs', None) 
        self.inv_loss = None
        self.eq_loss = None
        if 'paired' in  self.ssl_loss_str:
            loss_fn_inv_kwargs = loss_kwargs.get('loss_fn_inv_kwargs', None) 
            loss_fn_eq_kwargs = loss_kwargs.get('loss_fn_eq_kwargs', None) 
            loss_fn_inv = self.get_loss_fn(loss_kwargs['loss_fn_inv'], loss_fn_inv_kwargs)
            self.inv_loss_type = loss_kwargs['loss_fn_inv']
            loss_fn_eq = self.get_loss_fn(loss_kwargs['loss_fn_eq'], loss_fn_eq_kwargs)
            self.eq_loss_type = loss_kwargs['loss_fn_eq']
            paired_loss =  ssl_losses.__dict__[self.config['hparas']['ssl_loss']]
            ssl_loss = paired_loss(loss_fn_inv=loss_fn_inv, loss_fn_eq=loss_fn_eq, lmda=loss_kwargs['lmda'])
        else:
            ssl_loss = self.get_loss_fn(self.config['hparas']['ssl_loss'], loss_kwargs)
        return ssl_loss

In [7]:
!export HDF5_USE_FILE_LOCKING=FALSE


In [2]:


L.seed_everything(0)
## init config. Will be yaml eventually, but start as dict 

## init config. Will be yaml eventually, but start as dict 
config_path = "model_configs/barlow_audioset_resnet18_rand_crop.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)
config['num_workers'] = 3
config['hparas']['batch_size'] = 128
config['hparas']['global_batch_size'] = 128
config['num_gpus'] = 1 
model = LitAudioSetSSL(config)

[rank: 0] Seed set to 0


NameError: name 'LitAudioSetSSL' is not defined

In [ ]:
model = eval(config['module'])(config)

In [4]:
torch.set_float32_matmul_precision('medium')
torch.cuda.is_available()

True

In [5]:
#### Try module version 
from lightning_scripts.lightning_ssl_audioset import LitAudioSetSSL

L.seed_everything(0)
## init config. Will be yaml eventually, but start as dict 

## init config. Will be yaml eventually, but start as dict 
config_path = "model_configs/barlow_audioset_resnet18_rand_crop.yaml"
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)
config['num_workers'] = 3
config['hparas']['batch_size'] = 128
config['hparas']['global_batch_size'] = 128
config['num_gpus'] = 1 

module = eval(config['module'])(config)

[rank: 0] Seed set to 0


In [6]:
trainer = L.Trainer(
                    # callbacks=[lr_monitor],
                    # limit_train_batches=5,
                    limit_val_batches=2,
                    max_epochs=5,
                    gradient_clip_val=1, # clipt grad l2 norm to 1 

                    # callbacks=callbacks,
                    #  strategy='ddp_notebook',
                    #  reload_dataloaders_every_n_epochs=-1,
                    devices=1)
trainer.fit(module)

/mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /mnt/home/igriffith/envs/cochdnn_ssl_pl/lib/python3. ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name           | Type                       | Params | Mode 
----------------------------------------------------------------------
0 | transforms     | AudioCompose               | 0      | train
1 | audio_rep      | AudioToAudioRepresentation | 0      | train
2 | model          | ModelWithFrontEnd          | 24.0 M | train
3 | ssl_loss       | Barlow_Loss                | 0      | train
4 | class_loss     | BCEWithLogitsLoss          | 0      | train
5 | calc_precision | Bi

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined